# Implementing Transformers for Text Generation

### Import necessary labraries

In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.utils import get_file

### Load the Dataset

In [5]:
text = open("shakespeare.txt", 'rb').read().decode(encoding='utf-8')
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger f


### Preprocess the Dataset

In [6]:
vocab_size = 10000
seq_length = 100

# Create a TextVectorization layer to preprocess the text data
vectorize_layer = TextVectorization(max_tokens=vocab_size, output_mode='int')
text_ds = tf.data.Dataset.from_tensor_slices([text]).batch(1)
vectorize_layer.adapt(text_ds)

# Vectorize the text
vectorized_text = vectorize_layer([text])[0]
print("Vectorized text shape:", vectorized_text.shape)
print("First 10 vectorized tokens:", vectorized_text.numpy()[:10])

Vectorized text shape: (202646,)
First 10 vectorized tokens: [ 89 270 138  36 982 144 673 125  16 106]


### Create input and target sequences

In [11]:
# Function to generate input sequences and labels for training
def create_sequences(text, seq_length):
    input_seqs = []
    target_seqs = []
    for i in range(len(text) - seq_length):
        input_seq = (text[i:i + seq_length])
        target_seq = (text[i + 1 : i + seq_length + 1])
        input_seqs.append(input_seq)
        target_seqs.append(target_seq)

    return np.array(input_seq), np.array(target_seq)

In [12]:
# Generate sequences
X, y = create_sequences(vectorized_text.numpy(), seq_length)

# Check if sequences are correctly generated
print("Number of sequences generated:", len(X))
print("Sample input sequences:", X[0] if len(X) > 0 else "No sequences generated")

# Check if X and y are not empty
assert X.size > 0 , "input data X is empty"
assert y.size > 0 , "target data y is empty"
X = tf.convert_to_tensor(X)
y = tf.convert_to_tensor(y)
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Number of sequences generated: 100
Sample input sequences: 29
Shape of X: (100,)
Shape of y: (100,)


### Build the Transformer model

In [13]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding,MultiHeadAttention,Dense,LayerNormalization,Dropout

In [15]:
class TransformerBlock(tf.keras.layers.Layer):

    def __int__(self,embed_dim,num_heads,ff_dim,rate=0.1):
        super(TransformerBlock,self).__init__()
        self.att = MultiHeadAttention(num_heads=num_heads,key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim,activation="relu"),
            Dense(embed_dim),     
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self,inputs,training = False):
        attn_output = self.att(inputs,inputs)
        attn_output = self.dropout1(attn_output,training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output,training=training)
        return self.layernorm2(out1 + ffn_output)

class TransformerModel(Model):

    def __int__(self, vocab_size,embed_dim, num_heads,ff_dim, num_layers, seq_length):
        super(TransformerModel,self).__init__()
        self.embedding = Embedding(vocab_size,embed_dim)
        self.pos_encoding = self.positional_encoding(seq_length,embed_dim)
        self.transformer_blocks = [TransformerBlock(embed_dim,num_heads,ff_dim) for _ in range(num_layer)]
        self.dense = Dense(vocab_size)

    def positional_encoding(self,seq_length,embed_dim):
        angle_rads = self.get_angles(np.arange(seq_length)[:,np.newaxis],np.arange(embed_dim)[np.newaxis, :],embed_dim)
        angle_rads[:, 0::2] = np.sin(angle_rads[: , 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[: , 1::2])
        self.pos_encoding = angle_rads[np.newaxis, ...]
        return tf.cast(self.pos_encoding,dtype=tf.float32)

    def get_angles(self, pos,i,embed_dim):
        angle_rates = 1 / np.power(10000, (2* (i // 2)) / np.float32(embed_dim))
        return pos * angle_rates

    def call(self, inputs, training = False):
        seq_length = tf.shape(inputs)[1]
        x = self.embedding(inputs)
        x += self.pos_encoding[:, :seq_length, :]
        for transformer_block in self.transformer_blocks:
            x = transformer_blocks(x,training=training)

        output = self.dense(x)
        return output
        
        